# Exploration d'un modèle de Motion Flow
Ce notebook permet de charger une image statique de paysage, de générer un champ de mouvement synthétique (motion flow), et d'appliquer ce mouvement à l'image sans utiliser OpenCV pour sa lecture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
import os
from PIL import Image

# Fixer le seed pour la randomisation si nécessaire
np.random.seed(42)

## 1. Chargement de l'image de paysage (avec PIL et NumPy)
Nous chargeons une image locale présente dans le repository en utilisant `PIL` (Pillow) puis nous la convertissons en tableau (array) `NumPy`.

In [ ]:
# Chemin vers l'image dans votre projet.
image_path = '../data/landscape.jpg'  # <-- Modifiez avec le bon chemin !

if not os.path.exists(image_path):
    print(f"⚠️  Attention : L'image est introuvable au chemin '{image_path}'")
    print("Veuillez vérifier le nom et l'emplacement du fichier.")
    image = None
else:
    # Lecture de l'image avec PIL puis conversion en Numpy Array
    # Contrairement à OpenCV, PIL lit nativement en RGB, aucune conversion n'est nécessaire !
    pil_image = Image.open(image_path)
    image = np.array(pil_image)
    
    h, w, c = image.shape
    
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.title("Image Originale")
    plt.axis('off')
    plt.show()

## 2. Génération du champ de mouvement (Motion Flow)
Un *motion flow* ou flux optique peut être représenté par deux matrices de même taille que l'image : une pour le déplacement selon l'axe X (horizontal) et une pour l'axe Y (vertical).
Ici, nous allons générer un mouvement fluide et ondulatoire.

In [ ]:
if image is not None:
    # Génération de la grille de coordonnées
    x, y = np.meshgrid(np.arange(w), np.arange(h))

    # Création d'un champ de mouvement dynamique de type "vague"
    amplitude = 15.0
    frequency = 0.02

    # Le flux indique la direction et la force avec laquelle le pixel est déplacé
    flow_x = amplitude * np.sin(x * frequency) * np.cos(y * frequency)
    flow_y = amplitude * np.cos(x * frequency) * np.sin(y * frequency)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(flow_x, cmap='coolwarm')
    plt.title("Motion flow: Déplacement en X (U)")
    plt.colorbar()

    plt.subplot(1, 2, 2)
    plt.imshow(flow_y, cmap='coolwarm')
    plt.title("Motion flow: Déplacement en Y (V)")
    plt.colorbar()
    plt.show()
else:
    print("L'image n'étant pas chargée, cette étape est ignorée.")

## 3. Application du Motion Flow à l'image (Warping)
Puisque nous n'utilisons plus OpenCV, nous allons utiliser `scipy.ndimage.map_coordinates` pour appliquer la déformation par interpolation aux différentes couches de l'image (RGB).

In [ ]:
if image is not None:
    # La nouvelle source de couleur pour un pixel se trouve à sa position moins le vecteur de flow
    # Attention: scipy.ndimage utilise [y, x] au lieu de [x, y] pour les coordonnées
    map_y = y - flow_y
    map_x = x - flow_x
    
    warped_image = np.zeros_like(image)
    
    # map_coordinates s'applique canal par canal (R, G, B)
    for channel in range(c):
        warped_image[..., channel] = map_coordinates(
            image[..., channel], 
            [map_y, map_x], 
            order=1, # Ordre 1 = Interpolation bilinéaire
            mode='reflect' # Gère les bords comme OpenCV (BORDER_REFLECT)
        )

    plt.figure(figsize=(10, 6))
    plt.imshow(warped_image)
    plt.title("Image avec Motion Flow Appliqué (via SciPy)")
    plt.axis('off')
    plt.show()
else:
    print("L'image n'étant pas chargée, cette étape est ignorée.")

## Bonus : Création d'une animation GIF
Puisque le but du projet est d'animer l'image, générons plusieurs images (frames) en faisant varier la phase de notre onde au cours du temps.

In [ ]:
if image is not None:
    frames = []
    num_frames = 30

    for i in range(num_frames):
        # Evolution temporelle de la phase de la vague (boucle complète de 0 à 2*pi)
        phase = (i / num_frames) * 2 * np.pi
        f_x = amplitude * np.sin(x * frequency + phase) * np.cos(y * frequency)
        f_y = amplitude * np.cos(x * frequency) * np.sin(y * frequency + phase)
        
        m_y = y - f_y
        m_x = x - f_x
        
        frame = np.zeros_like(image)
        for channel in range(c):
            frame[..., channel] = map_coordinates(
                image[..., channel], 
                [m_y, m_x], 
                order=1,
                mode='reflect'
            )
            
        frames.append(Image.fromarray(frame))

    # Sauvegarde du GIF
    gif_path = 'animated_landscape.gif'
    frames[0].save(gif_path,
                   save_all=True,
                   append_images=frames[1:],
                   duration=100,
                   loop=0)

    print(f"Animation sauvegardée sous '{gif_path}' !")

    # Affichage dans le notebook
    from IPython.display import Image as IPyImage, display
    display(IPyImage(url=gif_path))
else:
    print("L'image n'étant pas chargée, cette étape est ignorée.")